# 01a KG Exploration for RxNorm data

## Imports

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import sys

project_root = os.chdir("..")
sys.path.insert(0, str(project_root))

os.getcwd()

In [ ]:
import pandas as pd
import json

In [ ]:
from src.knowledge_graph.openfda_connector import OpenFDAConnector
from src.knowledge_graph.rxnorm_connector import RxNormConnector
from src.knowledge_graph.rxnorm_extractor import RxNormExtractor

## Data

In [ ]:
# Test connection
rxnorm = RxNormConnector()
rxnorm_connected = rxnorm.connect()

if rxnorm_connected:
    rxnorm_result = rxnorm.test_query("aspirin")
    rxnorm.close()

In [ ]:
# Test extractor
rxnorm_connected = RxNormConnector()
rxnorm_connected.connect()

rxnorm_extractor = RxNormExtractor(rxnorm_connected.connection)

rxnrel_named_df = rxnorm_extractor.get_rxnrel_with_drug_names()

In [ ]:
rxnrel_named_df

In [ ]:
print(rxnrel_named_df.head())

In [ ]:
print(rxnrel_named_df.RELA.value_counts())

## Build KG

In [ ]:
KEEP_RELA = {
    "has_ingredient",
    "has_precise_ingredient",
    "has_tradename",
    "has_dose_form",
    "has_doseformgroup",
    "has_form",
    "has_part",
    "has_ingredients",
    "contains",
    "consists_of",
    "isa",
    "has_boss",
    "has_quantified_form",
    "reformulated_to",
}

DROP_RELA = {
    "ingredient_of",
    "precise_ingredient_of",
    "tradename_of",
    "dose_form_of",
    "doseformgroup_of",
    "form_of",
    "part_of",
    "ingredients_of",
    "contained_in",
    "constitutes",
    "inverse_isa",
    "boss_of",
    "quantified_form_of",
    "reformulation_of",
}

CANONICAL_RELA = {
    "isa",
    "has_ingredient",
    "has_precise_ingredient",
    "has_ingredients",
    "has_tradename",
    "has_dose_form",
    "has_doseformgroup",
    "has_form",
    "has_part",
    "contains",
    "consists_of",
    "has_boss",
    "has_quantified_form",
    "reformulated_to",
}


INVERSE_RELA = {
    "inverse_isa",
    "ingredient_of",
    "precise_ingredient_of",
    "ingredients_of",
    "tradename_of",
    "dose_form_of",
    "doseformgroup_of",
    "form_of",
    "part_of",
    "contained_in",
    "constitutes",
    "boss_of",
    "quantified_form_of",
    "reformulation_of",
}


DEFAULT_EXCLUDED_TTYS = {
    "SY",
    "TMSY",
    # I would start by excluding PSN too.
    # You can keep it if you want prescribable display names as nodes.
    "PSN",
}

In [ ]:
from typing import Optional, Iterable, Dict, Set
import pandas as pd
import networkx as nx
import logging

logger = logging.getLogger(__name__)


def rxnorm_tty_to_node_type(tty: Optional[str]) -> str:
    """
    Map RxNorm TTY values to simpler knowledge graph node categories.
    """
    if tty is None:
        return "unknown"

    tty = tty.upper()

    ingredient_ttys = {"IN", "PIN"}
    ingredient_group_ttys = {"MIN"}

    brand_ttys = {"BN"}

    clinical_drug_ttys = {"SCD"}
    branded_drug_ttys = {"SBD"}

    generic_abstraction_ttys = {
        "SCDC", "SCDF", "SCDG", "SCDGP", "SCDFP"
    }

    branded_abstraction_ttys = {
        "SBDC", "SBDF", "SBDG", "SBDFP"
    }

    dose_form_ttys = {"DF", "DFG"}

    pack_ttys = {"GPCK", "BPCK"}

    synonym_ttys = {"SY", "TMSY", "PSN"}

    if tty in ingredient_ttys:
        return "ingredient"
    if tty in ingredient_group_ttys:
        return "ingredient_group"
    if tty in brand_ttys:
        return "brand"
    if tty in clinical_drug_ttys:
        return "clinical_drug"
    if tty in branded_drug_ttys:
        return "branded_drug"
    if tty in generic_abstraction_ttys:
        return "clinical_drug_form"
    if tty in branded_abstraction_ttys:
        return "branded_drug_form"
    if tty in dose_form_ttys:
        return "dose_form"
    if tty in pack_ttys:
        return "pack"
    if tty in synonym_ttys:
        return "synonym"

    return "other"

In [ ]:
# Main graph builder
def build_rxnorm_knowledge_graph(
    rxnrel_named_df: pd.DataFrame,
    keep_rela: Optional[Set[str]] = None,
    exclude_rela: Optional[Set[str]] = None,
    exclude_ttys: Optional[Set[str]] = None,
    include_inverse_edges: bool = False,
) -> nx.MultiDiGraph:
    """
    Build a simple RxNorm knowledge graph from an enriched RXNREL DataFrame.

    Expected columns:
        RXCUI1
        RXCUI1_NAME
        RXCUI1_TTY
        RELA
        RXCUI2
        RXCUI2_NAME
        RXCUI2_TTY

    Graph design:
        - Each RXCUI is a node.
        - Node attributes include name, tty, and simplified node_type.
        - Each RXNREL relationship is a directed edge.
        - By default, inverse relationship duplicates are excluded.
        - By default, synonym-like TTYs are excluded as graph nodes.

    Args:
        rxnrel_named_df:
            DataFrame returned by get_rxnrel_with_drug_names().

        keep_rela:
            Optional set of relationship types to keep.
            If None, uses CANONICAL_RELA.

        exclude_rela:
            Optional set of relationship types to remove.
            If None and include_inverse_edges=False, removes INVERSE_RELA.

        exclude_ttys:
            Optional set of TTYs to exclude as nodes.
            If None, excludes SY, TMSY, PSN.

        include_inverse_edges:
            If True, keep inverse relationships too.
            If False, removes common inverse relationship duplicates.

    Returns:
        A NetworkX MultiDiGraph.
    """
    required_columns = {
        "RXCUI1",
        "RXCUI1_NAME",
        "RXCUI1_TTY",
        "RELA",
        "RXCUI2",
        "RXCUI2_NAME",
        "RXCUI2_TTY",
    }

    missing_columns = required_columns - set(rxnrel_named_df.columns)
    if missing_columns:
        raise ValueError(f"Missing required columns: {missing_columns}")

    df = rxnrel_named_df.copy()

    # Normalize core columns
    df["RXCUI1"] = df["RXCUI1"].astype(str)
    df["RXCUI2"] = df["RXCUI2"].astype(str)
    df["RELA"] = df["RELA"].astype(str)

    df["RXCUI1_TTY"] = df["RXCUI1_TTY"].astype(str).str.upper()
    df["RXCUI2_TTY"] = df["RXCUI2_TTY"].astype(str).str.upper()

    # Remove incomplete rows
    df = df.dropna(
        subset=[
            "RXCUI1",
            "RXCUI1_NAME",
            "RXCUI1_TTY",
            "RELA",
            "RXCUI2",
            "RXCUI2_NAME",
            "RXCUI2_TTY",
        ]
    )

    # Filter relationships
    if keep_rela is None:
        keep_rela = CANONICAL_RELA

    df = df[df["RELA"].isin(keep_rela)]

    if not include_inverse_edges:
        if exclude_rela is None:
            exclude_rela = INVERSE_RELA
        df = df[~df["RELA"].isin(exclude_rela)]

    # Filter node TTYs
    if exclude_ttys is None:
        exclude_ttys = DEFAULT_EXCLUDED_TTYS

    df = df[
        ~df["RXCUI1_TTY"].isin(exclude_ttys)
        & ~df["RXCUI2_TTY"].isin(exclude_ttys)
    ]

    # Build graph
    graph = nx.MultiDiGraph()

    for row in df.itertuples(index=False):
        rxcui1 = str(row.RXCUI1)
        rxcui2 = str(row.RXCUI2)

        rxcui1_tty = row.RXCUI1_TTY
        rxcui2_tty = row.RXCUI2_TTY

        graph.add_node(
            rxcui1,
            rxcui=rxcui1,
            name=row.RXCUI1_NAME,
            tty=rxcui1_tty,
            node_type=rxnorm_tty_to_node_type(rxcui1_tty),
        )

        graph.add_node(
            rxcui2,
            rxcui=rxcui2,
            name=row.RXCUI2_NAME,
            tty=rxcui2_tty,
            node_type=rxnorm_tty_to_node_type(rxcui2_tty),
        )

        graph.add_edge(
            rxcui2,
            rxcui1,
            rela=row.RELA,
            source="RXNREL",
        )

    logger.info(
        "Built RxNorm KG with %s nodes and %s edges",
        graph.number_of_nodes(),
        graph.number_of_edges(),
    )

    return graph

### Usage

In [ ]:
kg = build_rxnorm_knowledge_graph(rxnrel_named_df)

print(kg.number_of_nodes())
print(kg.number_of_edges())

In [ ]:
# Inspect a node
kg.nodes["38"]

In [ ]:
# Inspect outgoing relationships from one drug
test_rxcui = "38"
test_rxcui = "1760"

for source, target, data in kg.out_edges(test_rxcui, data=True):
    print(
        kg.nodes[source]["name"],
        data["rela"],
        kg.nodes[target]["name"],
        kg.nodes[target]["tty"]
    )


In [ ]:
def inspect_rela_tty_pairs(df: pd.DataFrame) -> pd.DataFrame:
    """
    Inspect relationship patterns using correct RxNorm direction:
    RXCUI2 --RELA--> RXCUI1
    """
    out = (
        df.groupby(["RXCUI2_TTY", "RELA", "RXCUI1_TTY"])
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
    )

    return out

In [ ]:
inspect_rela_tty_pairs(rxnrel_named_df).head(50)

In [ ]:
rela_tty_summary = inspect_rela_tty_pairs(rxnrel_named_df)

rela_tty_summary[
    rela_tty_summary["RELA"].isin([
        "has_ingredient",
        "has_tradename",
        "isa",
        "has_dose_form",
        "consists_of",
        "contains",
    ])
].head(100)

### Helper: inspect graph composition

In [ ]:
from collections import Counter


def summarize_rxnorm_graph(graph: nx.MultiDiGraph) -> None:
    """
    Print a simple summary of node types, TTYs, and relationship types.
    """
    node_types = Counter()
    tty_counts = Counter()
    rela_counts = Counter()

    for _, attrs in graph.nodes(data=True):
        node_types[attrs.get("node_type", "unknown")] += 1
        tty_counts[attrs.get("tty", "unknown")] += 1

    for _, _, attrs in graph.edges(data=True):
        rela_counts[attrs.get("rela", "unknown")] += 1

    print("Graph summary")
    print("-------------")
    print(f"Nodes: {graph.number_of_nodes():,}")
    print(f"Edges: {graph.number_of_edges():,}")

    print("\nNode types")
    for node_type, count in node_types.most_common():
        print(f"{node_type:25s} {count:,}")

    print("\nTTYs")
    for tty, count in tty_counts.most_common():
        print(f"{tty:10s} {count:,}")

    print("\nRelationships")
    for rela, count in rela_counts.most_common():
        print(f"{rela:30s} {count:,}")

In [ ]:
summarize_rxnorm_graph(kg)

### Helper: get a drug’s local neighborhood

In [ ]:
def get_drug_neighborhood(
    graph: nx.MultiDiGraph,
    rxcui: str,
    depth: int = 1,
) -> pd.DataFrame:
    """
    Return a DataFrame of relationships around a given RXCUI.

    Args:
        graph:
            RxNorm knowledge graph.

        rxcui:
            RXCUI to inspect.

        depth:
            Currently supports depth=1 for direct neighbors.

    Returns:
        DataFrame containing source, relationship, and target information.
    """
    if depth != 1:
        raise NotImplementedError("Only depth=1 is currently supported.")

    rxcui = str(rxcui)

    if rxcui not in graph:
        return pd.DataFrame()

    rows = []

    for source, target, attrs in graph.out_edges(rxcui, data=True):
        rows.append(
            {
                "direction": "outgoing",
                "source_rxcui": source,
                "source_name": graph.nodes[source].get("name"),
                "source_tty": graph.nodes[source].get("tty"),
                "rela": attrs.get("rela"),
                "target_rxcui": target,
                "target_name": graph.nodes[target].get("name"),
                "target_tty": graph.nodes[target].get("tty"),
            }
        )

    for source, target, attrs in graph.in_edges(rxcui, data=True):
        rows.append(
            {
                "direction": "incoming",
                "source_rxcui": source,
                "source_name": graph.nodes[source].get("name"),
                "source_tty": graph.nodes[source].get("tty"),
                "rela": attrs.get("rela"),
                "target_rxcui": target,
                "target_name": graph.nodes[target].get("name"),
                "target_tty": graph.nodes[target].get("tty"),
            }
        )

    return pd.DataFrame(rows)

In [ ]:
get_drug_neighborhood(kg, "38")

### Graph views

In [ ]:
kg_full = build_rxnorm_knowledge_graph(
    rxnrel_named_df,
    keep_rela=CANONICAL_RELA,
    exclude_ttys={"SY", "TMSY", "PSN"},
    include_inverse_edges=False,
)

In [ ]:
ingredient_rela = {
    "has_ingredient",
    "has_precise_ingredient",
    "has_ingredients",
    "consists_of",
    "contains",
}

kg_ingredients = build_rxnorm_knowledge_graph(
    rxnrel_named_df,
    keep_rela=ingredient_rela,
    exclude_ttys={"SY", "TMSY", "PSN", "DF", "DFG"},
    include_inverse_edges=False,
)

In [ ]:
brand_rela = {
    "has_tradename",
    "has_ingredient",
    "has_precise_ingredient",
}

kg_brands = build_rxnorm_knowledge_graph(
    rxnrel_named_df,
    keep_rela=brand_rela,
    exclude_ttys={"SY", "TMSY", "PSN", "DF", "DFG"},
    include_inverse_edges=False,
)

In [ ]:
taxonomy_rela = {
    "isa",
    "has_form",
    "has_dose_form",
    "has_doseformgroup",
}

kg_taxonomy = build_rxnorm_knowledge_graph(
    rxnrel_named_df,
    keep_rela=taxonomy_rela,
    exclude_ttys={"SY", "TMSY", "PSN"},
    include_inverse_edges=False,
)

In [ ]:
def export_graphml(graph: nx.MultiDiGraph, path: str) -> None:
    """
    Export graph to GraphML.

    GraphML is useful for visualization in Gephi, Cytoscape, and other graph tools.
    """
    nx.write_graphml(graph, path)

In [ ]:
export_graphml(kg_full, "rxnorm_kg.graphml")

In [ ]:
kg = build_rxnorm_knowledge_graph(
    rxnrel_named_df,
    keep_rela={
        "isa",
        "has_ingredient",
        "has_precise_ingredient",
        "has_ingredients",
        "has_tradename",
        "has_dose_form",
        "has_doseformgroup",
        "has_form",
        "consists_of",
        "contains",
    },
    exclude_ttys={
        "SY",
        "TMSY",
        "PSN",
    },
    include_inverse_edges=False,
)

summarize_rxnorm_graph(kg)

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

def plot_drug_neighborhood(graph, rxcui: str, depth: int = 1):
    """
    Plot a small neighborhood around one RXCUI.
    Best for inspection, not for the full RxNorm graph.
    """
    rxcui = str(rxcui)

    if rxcui not in graph:
        print(f"RXCUI {rxcui} not found in graph.")
        return

    nodes = {rxcui}

    if depth >= 1:
        nodes.update(graph.successors(rxcui))
        nodes.update(graph.predecessors(rxcui))

    subgraph = graph.subgraph(nodes).copy()

    pos = nx.spring_layout(subgraph, seed=42)

    labels = {
        node: subgraph.nodes[node].get("name", node)
        for node in subgraph.nodes
    }

    edge_labels = {
        (u, v): data.get("rela", "")
        for u, v, data in subgraph.edges(data=True)
    }

    plt.figure(figsize=(14, 10))
    nx.draw(
        subgraph,
        pos,
        labels=labels,
        with_labels=True,
        node_size=1800,
        font_size=8,
        arrows=True,
    )
    nx.draw_networkx_edge_labels(
        subgraph,
        pos,
        edge_labels=edge_labels,
        font_size=7,
    )
    plt.axis("off")
    plt.show()

In [ ]:
plot_drug_neighborhood(kg, "1760")

In [ ]:
from pyvis.network import Network
import networkx as nx

def export_drug_neighborhood_html(
    graph,
    rxcui: str,
    output_path: str = "drug_neighborhood.html",
    depth: int = 1,
):
    """
    Export a small interactive HTML visualization of one drug neighborhood.
    """
    rxcui = str(rxcui)

    if rxcui not in graph:
        raise ValueError(f"RXCUI {rxcui} not found in graph.")

    nodes = {rxcui}

    if depth >= 1:
        nodes.update(graph.successors(rxcui))
        nodes.update(graph.predecessors(rxcui))

    subgraph = graph.subgraph(nodes).copy()

    net = Network(
        height="750px",
        width="100%",
        directed=True,
        notebook=False,
    )

    for node_id, attrs in subgraph.nodes(data=True):
        label = attrs.get("name", node_id)
        title = (
            f"RXCUI: {attrs.get('rxcui', node_id)}<br>"
            f"Name: {attrs.get('name', '')}<br>"
            f"TTY: {attrs.get('tty', '')}<br>"
            f"Type: {attrs.get('node_type', '')}"
        )

        net.add_node(
            node_id,
            label=label,
            title=title,
            group=attrs.get("node_type", "unknown"),
        )

    for source, target, attrs in subgraph.edges(data=True):
        net.add_edge(
            source,
            target,
            label=attrs.get("rela", ""),
            title=attrs.get("rela", ""),
        )

    net.write_html(output_path)
    return output_path

In [ ]:
html_path = export_drug_neighborhood_html(
    kg,
    rxcui="1760",
    output_path="bromocriptine_neighborhood.html"
)

html_path